# Standard pipeline for evaluating models

Set up:
- Configure venv to run your model
- Download dataset from [Movie Lens 1M Dataset](https://grouplens.org/datasets/movielens/1m/) and store in `Dataset/ml-1m` folder
- Go through this notebook and replace # TODO areas with your code

In [ ]:
# Standard
import json
import os

# Third-party
import numpy as np
import pandas as pd

# Local
from Dataset.ml1m_data_loader import load_and_merge_data, ml_test_train_split
from evals.offline_eval_metrics import OfflineModelEvaluator, OfflineSlateEvaluator, coverage

# TODO - import model specific imports (preferably model class) here
from model.MODEL_TYPE.MODEL_code import MODEL

### Load dataset

In [ ]:
df = load_and_merge_data()
df_train, df_test = ml_test_train_split(df, test_proportion=0.2)

In [ ]:
df_train.head()

In [ ]:
df_test.head()

### Initialise and train model
This could be as a class which trains the model on intialisation or some other implementation so you can call it later for specific users and return Top-N or user specific slates.

In [ ]:
# TODO - set model name (e.g. NMF_MF) and load and train model on df_train
model_name = "MODEL_NAME_STRING"
model = MODEL(df_train)

### Evaluation metrics
Offline metrics - adapted from [here](https://github.com/aryan-jadon/Evaluation-Metrics-for-Recommendation-Systems/blob/main/recommenders/evaluation/python_evaluation.py)

Table from [here](https://github.com/recommenders-team/recommenders/blob/main/examples/03_evaluate/evaluation.ipynb)
|Metric|Range|Selection criteria|Limitation|Reference|
|------|-------------------------------|---------|----------|---------|
|RMSE|$> 0$|The smaller the better.|May be biased, and less explainable than MAE|[link](https://en.wikipedia.org/wiki/Root-mean-square_deviation)|
|MAE|$\geq 0$|The smaller the better.|Dependent on variable scale.|[link](https://en.wikipedia.org/wiki/Mean_absolute_error)|
|R2|$\leq 1$|The closer to $1$ the better.|Depend on variable distributions.|[link](https://en.wikipedia.org/wiki/Coefficient_of_determination)|
|Explained variance|$\leq 1$|The closer to $1$ the better.|Depend on variable distributions.|[link](https://en.wikipedia.org/wiki/Explained_variation)|

In [ ]:
model_eval = OfflineModelEvaluator()

In [ ]:
# TODO - Add code to extract predicted ratings for every movie in df_train
pred_ratings_train = MODEL.predictions_for_df_train

In [ ]:
# TODO - Add code to extract predicted ratings for every movie in df_test
pred_ratings_test = MODEL.predictions_for_df_train

In [ ]:
print("Metrics on training data:")
[print(f"{k}: {v:.2f}") for k, v in model_eval.calculate_metrics(list(df_train.rating), pred_ratings_train).items()]
print("\nMetrics on test data:")
[print(f"{k}: {v:.2f}") for k, v in model_eval.calculate_metrics(list(df_test.rating), pred_ratings_test).items()]

### Evaluate across subset of users in test dataset
Note: depending on how fast your modell runs, you may want to save out the metrics as you go along

In [ ]:
# Params:
N = 10
K = 10
min_rating_for_relevance = 3.5
users_to_eval = df_test["user_id"][:500]

In [ ]:
# Convert numpy types to Python types for JSON serialization
def convert_numpy_types(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


In [ ]:
results = []
all_recommendations = []
print(f"Evaluating recommendations for {len(users_to_eval)} users")
for user_id in users_to_eval:
    # Generate recommendation slate
    # TODO - update code to get reccommendations as list of movie_ids and
    #   predicted ratings for each of those from your model
    rec_ids, pred_ratings = MODEL.user_top_N(user_id)
    all_recommendations.append(rec_ids)

    # Run offline eval for this slate
    eval = OfflineSlateEvaluator(rec_ids, df_test, user_id, min_rating_for_relevance, "evals/movie_embeddings.pkl")
    results.append(eval.calculate_metrics(pred_ratings=pred_ratings, k=K))

# Calculate overall metrics
df_results = pd.DataFrame(results)
# Save results
output_dir = "evals/eval_results"
os.makedirs(output_dir, exist_ok=True) # Create output directory
df_results.to_csv(f'{output_dir}/user_metrics_{model_name}.csv', index=False)

overall_metrics = {
    'precision_at_k': df_results['precision_at_k'].mean(),
    'recall_at_k': df_results['recall_at_k'].mean(),
    'f1_at_k': df_results['f1_at_k'].mean(),
    'ndcg_at_k': df_results['ndcg_at_k'].mean(),
    'hit_rate_at_k': df_results['hit_rate_at_k'].mean(),
    'mean_average_precision': df_results['average_precision'].mean(),
    'mean_intra_list_similarity': df_results['intra_list_similarity'].mean(),
    'mean_gini_index': df_results['gini_index'].mean(),
    'catalog_coverage': coverage(all_recommendations, df_train['movie_id'].nunique()),
    'num_users': len(results),
    'k': K
}
with open(f'evals/eval_results/overall_metrics_{model_name}.json', 'w') as f:
    overall_metrics_serializable = {k: convert_numpy_types(v) for k, v in overall_metrics.items()}
    json.dump(overall_metrics, f, indent=2)

# Show output
for k, v in overall_metrics.items():
    print(f"{k}: {v:.2f}")

## User profile evaluation

|User|Total Ratings|Overall Avg Rating|Overall Std Dev|Must include| Should include| Must exclude|
|------|-------------------------------|---------|----------|---------|----|---|
| 6013 | 124 | 4.08 | 1.23 | Comedy, Drama | Musical, Romance | Action |
| 2195 | 258 | 3.41 | 1.41 | Action, Sci-Fi | Drama | Musical |
| 1198 | 102 | 3.66 | 1.51 | Action | Drama, Thriller | Children's |
| 3662 | 88  | 3.03 | 1.64 | Sci-Fi | Horror | Comedy |
| 4713 | 66  | 3.03 | 1.55 | Drama | Romance | Horror |


In [ ]:
user_ids = [6013, 2195, 1198, 3662, 4713]
for user_id in user_ids:
    print(f"Recommendations for user {user_id}")
    # TODO - update code to get reccommendations as list of movie_ids and
    #   predicted ratings for each of those from your model
    rec_ids, pred_ratings = MODEL.user_top_N(user_id)

    # TODO - Use this code or your own to show titles and genres and compare to the requirements above
    rec_titles = [df_train["title"][df_train.title[df_train.movie_id == id].index.to_list()[0]] for id in rec_ids]
    rec_genres = [df_train["genres"][df_train.genres[df_train.movie_id == id].index.to_list()[0]] for id in rec_ids]
    df = pd.DataFrame({"movie_id": rec_ids,
                       "title": rec_titles,
                       "genres": rec_genres})
    display(df)